In [3]:
import os
import csv
import string
import readability
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.colors
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from collections import defaultdict
from scipy import stats

In [4]:
Data_Root = '/kellogg/proj/hpk5609/Patent/Text/'

Test on subset

In [6]:
df_2001 = pd.read_csv(Data_Root+'Description_Text/pg_detail_desc_text_2001.tsv.zip', compression='zip', header=0, sep='\t', quotechar='"')


In [7]:
df_2001.head(3)

,pgpub_id,description_text,description_length
0,20010000001,31. The invention will be described in greater...,6584
1,20010000002,DETAILED DESCRIPTION OF THE INVENTION 65. The ...,29216
2,20010000003,DESCRIPTION OF THE PREFERRED EMBODIMENT 29. Th...,28479


In [17]:
len(df_2001)

56418

In [16]:
df_2001.loc[0]['description_text']

'31. The invention will be described in greater detail by way of specific examples. The following examples are offered for illustrative purposes, and are intended neither to limit or define the invention in any manner. \n EXAMPLE I 32. A solvent mixture in accordance with the invention was blended and added together to a standard vapor degreaser, the solvent mixture comprising: (I) about 90.0 wt. % n-propyl bromide; (ii) about 6 wt. % of a mixture of saturated terpenes, the terpene mixture comprising about 45 wt. % cis-pinane, about 45 wt. % trans-pinane, about 2 wt. % endo-isocamphene, about 2 wt. % &agr;-pinene, about 2 wt. % cis-para-menthane and about 2 wt. % trans-para-menthane; and (iii) about 4 wt. % of a mixture of low boiling solvents, the low boiling solvent mixture comprising about 0.5 wt. % nitromethane, about 0.5 wt. % 1,2-butylene oxide and about 3 wt. % 1,3-dioxolane. The thermostat on the vapor degreaser was adjusted to a temperature of about 160° F., and the system was

In [47]:
len(df_2001.loc[0]['description_text'].split())

1079

In [1]:
hype_dict = {
 'Importance': ['compelling',
 'critical',
 'crucial',
 'essential',
 'foundational',
 'fundamental',
 'imperative',
 'important',
 'indispensable',
 'invaluable',
 'key',
 'major',
 'paramount',
 'pivotal',
 'significant',
 'strategic',
 'timely',
 'ultimate',
 'urgent',
 'vital'],

'Novelty': ['creative',
 'emerging',
 'first',
 'groundbreaking',
 'innovative',
 'latest',
 'novel',
 'revolutionary',
 'unique',
 'unparalleled',
 'unprecedented'],
             
'Rigor': ['accurate',
 'advanced',
 'careful',
 'cohesive',
 'detailed',
 'nuanced',
 'powerful',
 'quality',
 'reproducible',
 'rigorous',
 'robust',
 'scientific',
 'sophisticated',
 'strong',
 'systematic'],
             
'Scale': ['ample',
 'biggest',
 'broad',
 'comprehensive',
 'considerable',
 'deeper',
 'diverse',
 'enormous',
 'expansive',
 'extensive',
 'fastest',
 'greatest',
 'huge',
 'immediate',
 'immense',
 'interdisciplinary',
 'international',
 'interprofessional',
 'largest',
 'massive',
 'multidisciplinary',
 'myriad',
 'overwhelming',
 'substantial',
 'top',
 'transdisciplinary',
 'tremendous',
 'vast'],
             
'Utility': ['accessible',
 'actionable',
 'deployable',
 'durable',
 'easy',
 'effective',
 'efficacious',
 'efficient',
 'generalizable',
 'ideal',
 'impactful',
 'intuitive',
 'meaningful',
 'productive',
 'ready',
 'relevant',
 'rich',
 'safer',
 'scalable',
 'seamless',
 'sustainable',
 'synergistic',
 'tailored',
 'tangible',
 'transformative',
 'user-friendly'],
             
'Quality': ['ambitious',
 'collegial',
 'dedicated',
 'exceptional',
 'experienced',
 'intellectual',
 'longstanding',
 'motivated',
 'premier',
 'prestigious',
 'promising',
 'qualified',
 'renowned',
 'senior',
 'skilled',
 'stellar',
 'successful',
 'talented',
 'vibrant'],
             
'Attitude': ['attractive',
 'confident',
 'exciting',
 'incredible',
 'interesting',
 'intriguing',
 'notable',
 'outstanding',
 'remarkable',
 'surprising'],
             
'Problem': ['alarming',
 'daunting',
 'desperate',
 'devastating',
 'dire',
 'dismal',
 'elusive',
 'stark',
 'unanswered',
 'unmet']
}

In [4]:
hype_words = set()
for cate in hype_dict:
    for w in hype_dict[cate]:
        hype_words.add(w)
hype_words_li = list(hype_words)

In [5]:
len(hype_words_li)

139

In [6]:
print(hype_words_li)

['pivotal', 'broad', 'outstanding', 'quality', 'transformative', 'productive', 'remarkable', 'foundational', 'fastest', 'accessible', 'sustainable', 'stellar', 'sophisticated', 'advanced', 'compelling', 'comprehensive', 'rigorous', 'ultimate', 'substantial', 'detailed', 'exceptional', 'unmet', 'synergistic', 'elusive', 'ample', 'durable', 'efficacious', 'essential', 'experienced', 'huge', 'cohesive', 'senior', 'important', 'interdisciplinary', 'easy', 'surprising', 'vibrant', 'interesting', 'innovative', 'extensive', 'seamless', 'overwhelming', 'scientific', 'nuanced', 'incredible', 'talented', 'tailored', 'daunting', 'impactful', 'imperative', 'novel', 'intuitive', 'scalable', 'deeper', 'unprecedented', 'strong', 'skilled', 'meaningful', 'unanswered', 'longstanding', 'groundbreaking', 'considerable', 'relevant', 'expansive', 'invaluable', 'crucial', 'enormous', 'intriguing', 'user-friendly', 'vast', 'collegial', 'multidisciplinary', 'biggest', 'qualified', 'confident', 'major', 'renow

In [5]:
table = str.maketrans('', '', string.punctuation)

def parse_text(text):
    # don't use split(' ')
    words = text.lower().split()
    # rm punctuation
    words = [w.translate(table) for w in words]
    return words


def count_hype_words(text):
    words = parse_text(text)
    mdict = defaultdict(int)
    for w in words:
        if w in hype_words:
            mdict[w] += 1
    cn_li = [mdict[w] for w in hype_words_li]
    return cn_li
    
def cal_FleschReadingEase(text):
    valid_words = parse_text(text)
    if len(valid_words) < 3:
        return np.nan
        
    measures = readability.getmeasures(text, lang='en')
    return measures['readability grades']['FleschReadingEase']

In [ ]:
years = list(range(2001, 2024))

for year in years:
    df_2001 = pd.read_csv(Data_Root+'Description_Text/pg_detail_desc_text_%d.tsv.zip'%year, compression='zip', header=0, sep='\t', quotechar='"')
    df_2001 = df_2001.loc[~df_2001['description_text'].isna()]
    df_2001['flesch_reading_ease'] = df_2001['description_text'].apply(cal_FleschReadingEase)
    df_2001['hwords_cn_li'] = df_2001['description_text'].apply(count_hype_words)
    tem_df = pd.DataFrame(df_2001['hwords_cn_li'].values.tolist(), index = df_2001.index, columns=hype_words_li)
    df_2001 = pd.concat([df_2001, tem_df], axis=1)
    df_2001['total_num_hype_words'] = df_2001[hype_words_li].sum(axis=1)
    df_2001['num_words'] = df_2001['description_text'].str.split().str.len()
    df_2001['frac_hype_words'] = df_2001['total_num_hype_words'] / df_2001['num_words']
    df_2001.drop(columns=['description_text', 'hwords_cn_li']).to_csv(Data_Root+'pgpub_id_text_measure_%d.csv'%year, header=True, index=False, quoting=csv.QUOTE_MINIMAL)

Check description text

In [6]:
df = pd.read_csv('/kellogg/proj/hpk5609/Patent/patent_reg_data.csv', header=0, low_memory=False, \
                 dtype={'pgpub_id': str, 'application_id': str, 'patent_id': str})

In [7]:
len(df)

2748927

In [8]:
ids = set(df['pgpub_id'].astype(int))

In [10]:
len(ids)

2748927

In [11]:
years = list(range(2001, 2024))
df_li = []
for year in years:
    df_ = pd.read_csv(Data_Root+'Description_Text/pg_detail_desc_text_%d.tsv.zip'%year, compression='zip', header=0, sep='\t', quotechar='"')
    df_ = df_.loc[df_['pgpub_id'].isin(ids)]
    df_.index = range(len(df_))
    df_li.append(df_)

In [12]:
combined_df = pd.concat(df_li, axis=0)
# combined_df = combined_df.reset_index(drop=True)
combined_df.index = range(len(combined_df))

In [13]:
len(combined_df)

2748927

In [14]:
del df_li

In [15]:
combined_df.head()

,pgpub_id,description_text,description_length
0,20130086733,DESCRIPTION OF EMBODIMENTS\n\nThe present appl...,34678
1,20130086756,BEST MODE FOR CARRYING OUT THE INVENTION\n\nTh...,28092
2,20130086760,DETAILED DESCRIPTION\n\nReference will now be ...,21317
3,20130086767,DETAILED DESCRIPTION\n\nFIG. 1shows a wiper bl...,11114
4,20130086781,DETAILED DESCRIPTION OF THE PREFERRED EMBODIME...,38508


In [16]:
combined_df.to_csv(Data_Root+'pgpub_id_fulltext.csv', header=True, index=False, quoting=csv.QUOTE_ALL)

In [14]:
pd.set_option('display.max_colwidth', None)

In [48]:
print(combined_df.loc[combined_df['pgpub_id'] == 20220222861, 'description_text'])

6940692    DETAILED DESCRIPTION OF THE EMBODIMENTS\n\nThe technical solutions in the embodiments of the present disclosure will be described below in conjunction with the drawings in the embodiments of the present disclosure. Obviously, the described embodiments are some of the embodiments of the present disclosure, but not all of the embodiments. Based on the embodiments in this disclosure, all other embodiments obtained by those of ordinary skill in the art without creative work shall fall within the scope of this disclosure.\n\nUnless otherwise specified, all technical and scientific terms used in the embodiments of the present disclosure have the same meaning as commonly understood by those skilled in the technical field of the present disclosure. The terms used in this disclosure are only for the purpose of describing specific embodiments and are not intended to limit the scope of the present disclosure.\n\nThe data encoding method provided by various embodiments of the present di

In [49]:
print(combined_df.loc[combined_df['pgpub_id'] == 20210227560, 'description_text'])

6535095    DETAILED DESCRIPTION OF ILLUSTRATIVE EMBODIMENTS\n\nTo improve rate experience of a cell edge user for NR, as shown inFIG. 1, a scenario in which a plurality of transmit/receive points (Tx/Rx point, TRP) are used as network devices is introduced. The transmit/receive point may be an antenna, a radio frequency unit, a femto base station, a pico base station, a micro base station, an urban base station, or the like. This is not specifically limited herein. In a typical scenario, a first TRP110sends first DCI to a terminal device120through a first physical downlink control channel (PDCCH), where the first DCI is used to indicate the terminal device to receive a first transport block on a first time-frequency resource; and a second TRP130sends second downlink control information DCI to the terminal device120through a second PDCCH, where the second DCI is used to indicate the terminal device to receive a second transport block on a second time-frequency resource. The first time-f

In [41]:
# 6130175 words (max length)
tem = combined_df.loc[combined_df['pgpub_id'] == 20140259212, 'description_text'].values[0]

In [47]:
tem[:5000]

'In a further embodiment, the present invention relates in paragraphs [0000.1.1.2] to [0514.1.1.2] to a further process for the production of at least a, preferably a, fine chemical selected from the group consisting of arginine, glutamate, glutamine and proline, as defined below and corresponding embodiments as described herein as follows\n\n[0001.1.1.2] to [0011.1.1.2] for the disclosure of these paragraphs see [0001.1.1.1] to [0011.1.1.1] above.\n\nIt is an object of the present invention to develop an inexpensive process for the synthesis of arginine, glutamate, glutamine and/or proline.\n\nfor the disclosure of this paragraph see [0013.1.1.1] above.\n\nAccordingly, in a first embodiment, the invention relates to a process for the production of at least one, preferably a, fine chemical selected from the group consisting of:\n\narginine, glutamate, glutamine and proline, or, in other words, of the “fine chemical” or “fine chemical of the invention”.\n\nThe terms “fine chemical of th

### Count We words

In [8]:
we_words = set(['we', 'us', 'our', 'ours'])

In [10]:
def count_total_we_words(text):
    words = parse_text(text)
    tot = 0
    for w in words:
        if w in we_words:
            tot += 1
    return tot

In [11]:
years = list(range(2001, 2024))

for year in years:
    df_2001 = pd.read_csv(Data_Root+'Description_Text/pg_detail_desc_text_%d.tsv.zip'%year, compression='zip', header=0, sep='\t', quotechar='"')
    df_2001 = df_2001.loc[~df_2001['description_text'].isna()]
    df_2001['total_num_we_words'] = df_2001['description_text'].apply(count_total_we_words)
    df_2001['num_words'] = df_2001['description_text'].str.split().str.len()
    df_2001['frac_we_words'] = df_2001['total_num_we_words'] / df_2001['num_words']
    df_2001.drop(columns=['description_text']).to_csv(Data_Root+'pgpub_id_we_measure_%d.csv'%year, header=True, index=False, quoting=csv.QUOTE_MINIMAL)
    